## Gold Preprocessing (Deliverable 1.3.1)

This notebook completes the Gold-layer preprocessing stage of the Medallion Architecture. It prepares the pump sensor dataset for model training and for the comparative evaluation described in Section C of the project proposal.

**Purpose:**  
To transform the cleaned Silver-layer dataset into a fully model-ready Gold dataset using the final feature registry, imputation decisions, and anomaly labels produced during Silver EDA. This ensures that both the baseline model and the three-stage cascade model are trained on a consistent and reproducible feature set.

**Key Goals:**

- Load the finalized Silver dataset and feature registry.
- Apply the Silver EDA imputation strategy (forward-fill followed by median).
- Standardize and scale feature values as required for model stability.
- Construct the model-ready Gold dataframe with:
  - 50 vetted numeric features for Stage 1 (broad) modeling,
  - A reduced feature subset for Stage 2 (narrow) modeling,
  - Profile- and rule-based sensor groupings for Stage 3 confirmation logic.
- Generate and export all Gold-layer preprocessing artifacts, including:
  - The Gold preprocessed parquet dataset,
  - Stage 1 and Stage 2 feature sets,
  - Stage 3 rule/profile sensor lists,
  - A preprocessing summary and metadata record.

**Relevance to Section C:**  
Outputs from this notebook directly support the methods described in Section C by:

- Providing a stable, aligned feature matrix for the baseline Isolation Forest and the three-stage cascade (C.2, C.2.A).
- Ensuring consistent preprocessing necessary for the paired model comparison and alert-quality evaluation (C.4).
- Supplying the structured Gold dataset that underpins the visual communication of alert patterns and model differences (C.6).

This notebook finalizes the dataset that the Gold Modeling notebook will use to implement, evaluate, and compare both anomaly-detection approaches.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Sequence, Tuple, Union

#import os
#import glob

from pathlib import Path
import yaml
import re

import logging
import wandb

import pandas as pd 
import numpy as np 

import matplotlib.pyplot as plt 
import seaborn as sns 

import joblib 


from sklearn.model_selection import train_test_split, KFold

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans


import pyarrow.parquet as pq
import pyarrow as pa


import hashlib


# Custom Utilities Module
from utils.paths import get_paths
from utils.file_io import load_data, save_data, save_json, load_json
from utils.eda_logging import profile_dataframe
from utils.logging_setup import configure_logging
from utils.wandb_utils import finalize_wandb_stage

# Ledger 
from utils.ledger import Ledger

# Show more columns
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)


In [90]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [91]:
def log_gold_paths(paths, logger: logging.Logger) -> None:
    logger.info("Project Root Path Loaded: %s", paths.root)
    logger.info("Project Logging Path Loaded: %s", paths.logs)
    logger.info("Project Artifacts Path Loaded: %s", paths.artifacts)
    logger.info("Project Notebooks Path Loaded %s", paths.notebooks)
    logger.info("Project Data Path Loaded: %s", paths.data)
    logger.info("Data Bronze Path Loaded: %s", paths.data_bronze)
    logger.info("Data Bronze Training Path Loaded: %s", paths.data_bronze_train)
    logger.info("Data Bronze Testing Path Loaded: %s", paths.data_bronze_test)
    logger.info("Data Silver Path Loaded: %s", paths.data_silver)
    logger.info("Data Silver Training Path Loaded: %s", paths.data_silver_train)
    logger.info("Data Silver Testing Path Loaded: %s", paths.data_silver_test)
    logger.info("Data Gold Path Loaded: %s", paths.data_gold)

In [92]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# Configurables

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Stage Details
STAGE = "gold"
LAYER_NAME = "gold"
GOLD_VERSION = "gold__001"
RECIPE_ID = "gold_preprocessing__v001_cascade"


DATASET_NAME_CONFIG = "pump"
DATASET_NAME = str(DATASET_NAME_CONFIG).strip().lower()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Weights and Biases
WANDB_PROJECT = "capstone"
WANDB_ENTITY = "dcoo230-western-governors-university"
WANDB_RUN_NAME = f"{GOLD_VERSION}"


#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# File names
SILVER_FILE_NAME = f"{DATASET_NAME}__silver__train.parquet"

GOLD_FILE_NAME = f"{DATASET_NAME}__gold__preprocessed.parquet"
GOLD_TRAIN_FILE_NAME = f"{DATASET_NAME}__gold__train.parquet"
GOLD_TEST_FILE_NAME = f"{DATASET_NAME}__gold__test.parquet"
GOLD_FIT_FILE_NAME = f"{DATASET_NAME}__gold__fit_normal_only.parquet"

GOLD_PRESCALED_FILE_NAME = f"{DATASET_NAME}__gold__preprocessed_prescaled.parquet"
GOLD_SCALED_FILE_NAME = f"{DATASET_NAME}__gold__preprocessed_scaled.parquet"

FEATURE_REGISTRY_FILE_NAME = f"{DATASET_NAME}__silver__feature_registry.json"
IMPUTE_RECOMMENDATION_FILE_NAME = "imputation__recommendation.json"

STAGE1_FEATURES_FILE_NAME = f"{DATASET_NAME}__gold__stage1_features.json"
STAGE2_FEATURES_FILE_NAME = f"{DATASET_NAME}__gold__stage2_features.json"
STAGE3_PRIMARY_FILE_NAME = f"{DATASET_NAME}__gold__stage3_primary_rule_sensors.json"
STAGE3_SECONDARY_FILE_NAME = f"{DATASET_NAME}__gold__stage3_secondary_rule_sensors.json"
CASCADE_RESULTS_FILE_NAME = f"{DATASET_NAME}__gold__cascade_results.csv"
COMPARISON_FILE_NAME = f"{DATASET_NAME}__gold__comparison__baseline_vs_cascade.csv"

GOLD_PREPROCESSING_LEDGER_FILE_NAME = f"ledger__{DATASET_NAME}__gold_preprocessing.json"

# Preprocessing Summaries
PREPROCESSING_SUMMARY_FILE_NAME = f"{DATASET_NAME}__gold__preprocessing_summary.json"
PREPROCESSING_METADATA_FILE_NAME = f"{DATASET_NAME}__gold__preprocessing_metadata.json"
REFERENCE_PROFILE_FILE_NAME = f"{DATASET_NAME}__gold__reference_profile.csv"
#PREPROCESSOR_SCALER_FILE_NAME = f"{DATASET_NAME}__gold__preprocessor_scaler.joblib"




#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Split configuration
TRAIN_FRACTION = 0.7
RANDOM_SEED = 42

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Scaling posture
#USE_ROBUST_SCALER = False

# options: "robust", "standard", "minmax"
SCALER_KIND = "robust"  

# Don't use an f string here, .format() will fill in the the items
SCALER_ARTIFACT_NAME_TEMPLATE = "{dataset}__gold__{scaler_kind}_scaler.joblib"




#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

# Stage 2 / Stage 3 sizing
STAGE2_TARGET_FEATURE_COUNT = 12
STAGE3_PRIMARY_COUNT = 5
STAGE3_SECONDARY_COUNT = 8

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 


In [135]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# Paths Setup

# Get Path's Object
paths = get_paths()

# Paths

# Silver
SILVER_TRAIN_DATA_PATH = paths.data_silver_train / SILVER_FILE_NAME
#SILVER_ARTIFACTS_PATH = paths.artifacts / "silver" / DATASET_NAME
SILVER_ARTIFACTS_PATH = paths.artifacts / "silver" / DATASET_NAME
SILVER_EDA_ARTIFACTS_PATH = paths.artifacts / "silver_eda" / DATASET_NAME

# Gold
GOLD_DATA_PATH = paths.data_gold / GOLD_FILE_NAME
GOLD_TRAIN_DATA_PATH = paths.data_gold / GOLD_TRAIN_FILE_NAME
GOLD_TEST_DATA_PATH = paths.data_gold / GOLD_TEST_FILE_NAME
GOLD_FIT_DATA_PATH = paths.data_gold / GOLD_FIT_FILE_NAME

GOLD_PRESCALED_DATA_PATH = paths.data_gold / GOLD_PRESCALED_FILE_NAME
GOLD_SCALED_DATA_PATH = paths.data_gold / GOLD_SCALED_FILE_NAME

GOLD_ARTIFACTS_PATH = paths.artifacts / "gold" / DATASET_NAME

# Jsons
FEATURE_REGISTRY_PATH = SILVER_ARTIFACTS_PATH / FEATURE_REGISTRY_FILE_NAME
IMPUTE_RECOMMENDATION_PATH = SILVER_EDA_ARTIFACTS_PATH / IMPUTE_RECOMMENDATION_FILE_NAME

# Stages
STAGE1_FEATURES_PATH = GOLD_ARTIFACTS_PATH / STAGE1_FEATURES_FILE_NAME
STAGE2_FEATURES_PATH = GOLD_ARTIFACTS_PATH / STAGE2_FEATURES_FILE_NAME
STAGE3_PRIMARY_PATH = GOLD_ARTIFACTS_PATH / STAGE3_PRIMARY_FILE_NAME
STAGE3_SECONDARY_PATH = GOLD_ARTIFACTS_PATH / STAGE3_SECONDARY_FILE_NAME
CASCADE_RESULTS_PATH = GOLD_ARTIFACTS_PATH / CASCADE_RESULTS_FILE_NAME
COMPARISON_PATH = GOLD_ARTIFACTS_PATH / COMPARISON_FILE_NAME

# Preprocessing Summaries
PREPROCESSING_SUMMARY_PATH = GOLD_ARTIFACTS_PATH / PREPROCESSING_SUMMARY_FILE_NAME
PREPROCESSING_METADATA_PATH = GOLD_ARTIFACTS_PATH / PREPROCESSING_METADATA_FILE_NAME
REFERENCE_PROFILE_PATH = GOLD_ARTIFACTS_PATH / REFERENCE_PROFILE_FILE_NAME
#PREPROCESSOR_SCALER_PATH = GOLD_ARTIFACTS_PATH / PREPROCESSOR_SCALER_FILE_NAME



# Logs
LOGS_PATH = paths.logs

# Path Failsafes
GOLD_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
GOLD_ARTIFACTS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)




In [137]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [138]:
# Logging Setup

# Create gold log path 
gold_log_path = paths.logs / "gold_preprocessing.log"

# Initial Logger
configure_logging(
    "capstone",
    gold_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.gold")

# Log load and initiation
logger.info("Gold stage starting")

# Log paths loads
log_gold_paths(paths, logger)


2026-03-03 20:25:19,368 | INFO | capstone.gold | Gold stage starting
2026-03-03 20:25:19,380 | INFO | capstone.gold | Project Root Path Loaded: /workspace
2026-03-03 20:25:19,387 | INFO | capstone.gold | Project Logging Path Loaded: /workspace/logs
2026-03-03 20:25:19,393 | INFO | capstone.gold | Project Artifacts Path Loaded: /workspace/artifacts
2026-03-03 20:25:19,397 | INFO | capstone.gold | Project Notebooks Path Loaded /workspace/notebooks
2026-03-03 20:25:19,399 | INFO | capstone.gold | Project Data Path Loaded: /workspace/data
2026-03-03 20:25:19,401 | INFO | capstone.gold | Data Bronze Path Loaded: /workspace/data/bronze
2026-03-03 20:25:19,407 | INFO | capstone.gold | Data Bronze Training Path Loaded: /workspace/data/bronze/train
2026-03-03 20:25:19,412 | INFO | capstone.gold | Data Bronze Testing Path Loaded: /workspace/data/bronze/test
2026-03-03 20:25:19,415 | INFO | capstone.gold | Data Silver Path Loaded: /workspace/data/silver
2026-03-03 20:25:19,419 | INFO | capstone.g

In [139]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# W&B

wandb_run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=WANDB_RUN_NAME,
    job_type="gold_preprocessing",
    config={
        "gold_version": GOLD_VERSION,
        "dataset": DATASET_NAME,
        "stage": STAGE,
        "train_fraction": TRAIN_FRACTION,
        "silver_path": str(SILVER_TRAIN_DATA_PATH),
        "feature_registry_path": str(FEATURE_REGISTRY_PATH),
        "impute_recommendation_path": str(IMPUTE_RECOMMENDATION_PATH),
        "gold_output_path": str(GOLD_DATA_PATH),
        "scaler_kind": str(SCALER_KIND),
        "stage2_target_feature_count": int(STAGE2_TARGET_FEATURE_COUNT),
        "stage3_primary_count": int(STAGE3_PRIMARY_COUNT),
        "stage3_secondary_count": int(STAGE3_SECONDARY_COUNT),
    },
)
logger.info("W&B initialized: %s", wandb.run.name)


2026-03-03 20:25:21,222 | INFO | capstone.gold | W&B initialized: gold__001


In [141]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [142]:
# Ledger Setup

ledger = Ledger(stage=STAGE, recipe_id=RECIPE_ID)

ledger.add(
    kind="step",
    step="init",
    message="Initialized ledger",
    logger=logger
)


2026-03-03 20:25:22,009 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-03T20:25:22.009849+00:00', 'stage': 'gold', 'recipe': 'gold_preprocessing__v001_cascade', 'kind': 'step', 'step': 'init', 'message': 'Initialized ledger', 'why': None, 'consequence': None, 'data': {}}


{'ts_utc': '2026-03-03T20:25:22.009849+00:00',
 'stage': 'gold',
 'recipe': 'gold_preprocessing__v001_cascade',
 'kind': 'step',
 'step': 'init',
 'message': 'Initialized ledger',
 'why': None,
 'consequence': None,
 'data': {}}

In [143]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [144]:
silver_path = SILVER_TRAIN_DATA_PATH
feature_registry_path = FEATURE_REGISTRY_PATH
imputation_recommendation_path = IMPUTE_RECOMMENDATION_PATH

logger.info("Loading Silver parquet: %s", silver_path)
silver_dataframe = load_data(silver_path)

logger.info("Loading feature registry: %s", feature_registry_path)
feature_registry = load_json(feature_registry_path)
feature_columns = feature_registry.get("feature_columns", [])
feature_set_id = feature_registry.get("feature_set_id", "unknown_feature_set")

logger.info("Loading imputation recommendation: %s", imputation_recommendation_path)
imputation_recommendation = load_json(imputation_recommendation_path, raise_if_missing=False, default={})
recommended_imputation = imputation_recommendation.get("recommendation", "global_median")

logger.info("Silver shape=%s", silver_dataframe.shape)
logger.info("Feature count=%d", len(feature_columns))
logger.info("Recommended imputation=%s", recommended_imputation)

#### #### #### #### #### #### #### #### 

ledger.add(
    kind="step",
    step="load_inputs",
    message="Loaded Silver parquet, feature registry, and imputation recommendation.",
    data={
        "silver_path": str(silver_path),
        "feature_registry_path": str(feature_registry_path),
        "impute_recommendation_path": str(imputation_recommendation_path),
        "feature_count": int(len(feature_columns)),
        "feature_set_id": str(feature_set_id),
        "recommended_imputation": str(recommended_imputation),
        "shape": {"rows": int(len(silver_dataframe)), "cols": int(len(silver_dataframe.columns))},
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 

silver_dataframe.head(3)

2026-03-03 20:25:22,841 | INFO | capstone.gold | Loading Silver parquet: /workspace/data/silver/train/pump__silver__train.parquet
2026-03-03 20:25:22,845 | INFO | capstone.file_io | Loading Parquet: /workspace/data/silver/train/pump__silver__train.parquet
2026-03-03 20:25:24,429 | INFO | capstone.gold | Loading feature registry: /workspace/artifacts/silver/pump/pump__silver__feature_registry.json
2026-03-03 20:25:24,438 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/silver/pump/pump__silver__feature_registry.json
2026-03-03 20:25:24,462 | INFO | capstone.gold | Loading imputation recommendation: /workspace/artifacts/silver_eda/pump/imputation__recommendation.json
2026-03-03 20:25:24,469 | INFO | capstone.file_io | Loading JSON: /workspace/artifacts/silver_eda/pump/imputation__recommendation.json
2026-03-03 20:25:24,487 | INFO | capstone.gold | Silver shape=(220320, 85)
2026-03-03 20:25:24,489 | INFO | capstone.gold | Feature count=50
2026-03-03 20:25:24,491 | INFO | cap

,meta__asset_id,meta__cleaning_recipe_id,meta__dataset,meta__dataset_name,meta__dataset_source,meta__event_id,meta__event_time_parse_success_percent,meta__event_time_source,meta__feature_count,meta__feature_set_id,meta__has_label_candidates,meta__has_status_candidates,meta__ingested_at_utc,meta__label_source,meta__label_source_column,meta__label_source_kind,meta__label_type,meta__layer,meta__processed_at_utc,meta__record_id,meta__run_id,meta__silver_version,meta__source_file,meta__source_row_id,meta__split,event_time,event_step,time_index,event_date,anomaly_flag,is_anomaly,is_normal,status_normal_value,sensor_00,sensor_01,sensor_02,sensor_03,sensor_04,sensor_05,sensor_06,sensor_07,sensor_08,sensor_09,sensor_10,sensor_11,sensor_12,sensor_13,sensor_14,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,sensor_22,sensor_23,sensor_24,sensor_25,sensor_26,sensor_27,sensor_28,sensor_29,sensor_30,sensor_31,sensor_32,sensor_33,sensor_34,sensor_35,sensor_36,sensor_37,sensor_38,sensor_39,sensor_40,sensor_41,sensor_42,sensor_43,sensor_44,sensor_45,sensor_46,sensor_47,sensor_48,sensor_49,sensor_51,timestamp,machine_status
0,asset__001,silver__clean__dataset__agnostic__v001,pump,pump,meta__dataset,pump:asset__001:run__001:0,100.0,timestamp,50,feature_set__bea5fdd737ae53fcd80ca84cafcd0c40,0,1,2026-03-01 03:54:51.311816+00:00,<NA>,machine_status,status,<NA>,silver,2026-03-01 04:18:23.781532+00:00,14598431322315673869,run__001,silver__001,sensor.csv,0,unsplit,2018-04-01 00:00:00+00:00,0,0,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,2.465394,47.09201,53.2118,46.31076,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,37.22740,47.52422,31.11716,1.681353,419.5747,461.8781,466.3284,2.565284,665.3993,398.9862,880.0001,498.8926,975.9409,627.6740,741.7151,848.0708,429.0377,785.1935,684.9443,594.4445,682.8125,680.4416,433.7037,171.9375,341.9039,195.0655,90.32386,40.36458,31.51042,70.57291,30.98958,31.770832,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,201.3889,2018-04-01 00:00:00,NORMAL
1,asset__001,silver__clean__dataset__agnostic__v001,pump,pump,meta__dataset,pump:asset__001:run__001:1,100.0,timestamp,50,feature_set__bea5fdd737ae53fcd80ca84cafcd0c40,0,1,2026-03-01 03:54:51.311816+00:00,<NA>,machine_status,status,<NA>,silver,2026-03-01 04:18:23.781532+00:00,15954729095895098000,run__001,silver__001,sensor.csv,1,unsplit,2018-04-01 00:01:00+00:00,1,1,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,2.465394,47.09201,53.2118,46.31076,634.3750,76.45975,13.41146,16.13136,15.56713,15.05353,37.22740,47.52422,31.11716,1.681353,419.5747,461.8781,466.3284,2.565284,665.3993,398.9862,880.0001,498.8926,975.9409,627.6740,741.7151,848.0708,429.0377,785.1935,684.9443,594.4445,682.8125,680.4416,433.7037,171.9375,341.9039,195.0655,90.32386,40.36458,31.51042,70.57291,30.98958,31.770832,41.92708,39.641200,65.68287,50.92593,38.194440,157.9861,67.70834,201.3889,2018-04-01 00:01:00,NORMAL
2,asset__001,silver__clean__dataset__agnostic__v001,pump,pump,meta__dataset,pump:asset__001:run__001:2,100.0,timestamp,50,feature_set__bea5fdd737ae53fcd80ca84cafcd0c40,0,1,2026-03-01 03:54:51.311816+00:00,<NA>,machine_status,status,<NA>,silver,2026-03-01 04:18:23.781532+00:00,10041703297090838359,run__001,silver__001,sensor.csv,2,unsplit,2018-04-01 00:02:00+00:00,2,2,2018-04-01 00:00:00+00:00,0,0,1,NORMAL,2.444734,47.35243,53.2118,46.39757,638.8889,73.54598,13.32465,16.03733,15.61777,15.01013,37.86777,48.17723,32.08894,1.708474,420.8480,462.7798,459.6364,2.500062,666.2234,399.9418,880.4237,501.3617,982.7342,631.1326,740.8031,849.8997,454.2390,778.5734,715.6266,661.5740,721.8750,694.7721,441.2635,169.9820,343.1955,200.9694,93.90508,41.40625,31.25000,69.53125,30.46875,31.770830,41.66666,39.351852,65.39352,51.21528,38.194443,155.9606,67.12963,203.7037,2018-04-01 00:02:00,NORMAL


In [145]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
dataframe = silver_dataframe.copy()

#### #### #### #### #### #### #### #### 

# Add Layer Tag
dataframe["meta__layer"] = LAYER_NAME

# Get Current Timestamp
processed_at_utc = pd.Timestamp.now(tz="UTC")

# Add Gold Processing Column and timestamp start
dataframe["meta__gold__processed_at_utc"] = processed_at_utc

# Add Silver Version Meta Column
dataframe["meta__gold_version"] = GOLD_VERSION

# Add Silver Cleaning Recipe ID Meta Column
if "meta__preprocessing_recipe_id" not in dataframe.columns:
    dataframe["meta__preprocessing_recipe_id"] = pd.NA
dataframe["meta__preprocessing_recipe_id"] = RECIPE_ID


#### #### #### #### #### #### #### #### 

gold_working_dataframe = dataframe.copy()


In [147]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [148]:
def build_episode_based_split_mask(
    dataframe: pd.DataFrame,
    *,
    train_fraction: float,
    episode_column: str = "meta__episode_id",
) -> tuple[pd.Series, dict]:
    """
    Build a train/test mask at the episode level.

    - Unique episodes are sorted by their id (which should already follow time order).
    - The earliest `train_fraction` of episodes are assigned to train.
    - All rows in those episodes are train; the rest are test.
    """
    if episode_column not in dataframe.columns:
        raise ValueError(f"Episode column '{episode_column}' not found in dataframe.")

    unique_episodes = np.sort(dataframe[episode_column].dropna().unique())
    n_episodes = len(unique_episodes)

    if n_episodes == 0:
        raise ValueError("No episodes found to build a split mask.")

    train_episode_count = max(1, int(np.floor(n_episodes * train_fraction)))
    train_episodes = set(unique_episodes[:train_episode_count])

    train_mask = dataframe[episode_column].isin(train_episodes)

    split_info = {
        "train_fraction": float(train_fraction),
        "episode_column": episode_column,
        "total_episodes": int(n_episodes),
        "train_episode_count": int(train_episode_count),
        "train_episodes": [int(e) for e in sorted(train_episodes)],
        "train_rows": int(train_mask.sum()),
        "test_rows": int((~train_mask).sum()),
    }

    return train_mask, split_info

In [149]:
train_mask, split_info = build_episode_based_split_mask(
    gold_working_dataframe,
    train_fraction=TRAIN_FRACTION,
    episode_column="meta__episode_id",
)

#### #### #### #### #### #### #### #### 

ledger.add(
    kind="step",
    step="build_episode_based_split_mask",
    message="Created time-ordered, episodic sorted train/test split for Gold modeling.",
    data=split_info,
    logger=logger,
)

#### #### #### #### #### #### #### #### 

split_info

AttributeError: 'function' object has no attribute 'columns'

In [ ]:
def stamp_training_metadata(
    dataframe: pd.DataFrame,
    train_mask: pd.Series | np.ndarray | None,
) -> pd.DataFrame:
    """
    If train_mask is provided, stamp:
      - meta__is_train_flag : bool
      - meta__is_train      : 'yes' / 'no'
      - meta__train_mask    : 'train' / 'test'

    Returns a new dataframe (does not modify in place).
    """
    working_dataframe = dataframe.copy()

    if train_mask is None:
        # No split defined yet; just return unchanged
        return working_dataframe

    # Align mask to dataframe index if it's a Series
    if isinstance(train_mask, pd.Series):
        # Safety: align on index
        train_mask_aligned = working_dataframe.index.to_series().map(train_mask).fillna(False).astype(bool)
    else:
        # Assume numpy array, same order/length as dataframe
        if len(train_mask) != len(working_dataframe):
            raise ValueError("train_mask length does not match dataframe length")
        train_mask_aligned = pd.Series(train_mask, index=working_dataframe.index, dtype=bool)

    # 1) Boolean flag 
    working_dataframe["meta__is_train_flag"] = train_mask_aligned

    # 2) Human-readable yes/no
    working_dataframe["meta__is_train"] = np.where(train_mask_aligned, "yes", "no")

    # 3) Explicit “train/test” label for debugging/reports
    working_dataframe["meta__train_mask"] = np.where(train_mask_aligned, "train", "test")

    return working_dataframe

In [ ]:
gold_working_dataframe = stamp_training_metadata(
    gold_working_dataframe,
    train_mask=train_mask,
)

#### #### #### #### #### #### #### #### 

ledger.add(
    kind="step",
    step="add_meta__train_mask_stamping",
    message="Stamping training mask to meta for tracking.",
    data=split_info,
    logger=logger,
)

#### #### #### #### #### #### #### #### 

In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
def select_numeric_feature_columns(
    dataframe: pd.DataFrame,
    *,
    feature_columns: list[str],
) -> list[str]:
    numeric_feature_columns: list[str] = []

    for feature_name in feature_columns:
        if feature_name not in dataframe.columns:
            continue
        if pd.api.types.is_numeric_dtype(dataframe[feature_name]):
            numeric_feature_columns.append(feature_name)

    return numeric_feature_columns


In [ ]:
numeric_feature_columns = select_numeric_feature_columns(
    silver_dataframe,
    feature_columns=feature_columns,
)


#### #### #### #### #### #### #### #### 

# TODO: Need Logger
# TODO: Need Ledger

#### #### #### #### #### #### #### #### 


In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
def apply_imputation(
    dataframe: pd.DataFrame,
    *,
    numeric_feature_columns: list[str],
    method: str,
    train_mask: pd.Series | None = None,
) -> tuple[pd.DataFrame, dict]:
    
    # Copy Dataframe
    working_dataframe = dataframe.copy()

    # Decide which rows define the stats ("fit" rows)
    if train_mask is not None:
        stats_dataframe = working_dataframe.loc[train_mask].copy()
    else:
        stats_dataframe = working_dataframe

    # Decide Fill method
    # Fill Method - Foward Fill within group with median
    if method == "forward_fill_within_group_then_median":
        grouping_columns: list[str] = []

        if "meta__asset_id" in working_dataframe.columns:
            grouping_columns.append("meta__asset_id")
        if "meta__run_id" in working_dataframe.columns:
            grouping_columns.append("meta__run_id")

        ordering_column = None
        if "event_step" in working_dataframe.columns:
            ordering_column = "event_step"
        elif "time_index" in working_dataframe.columns:
            ordering_column = "time_index"

        if len(grouping_columns) > 0 and ordering_column is not None:
            working_dataframe = working_dataframe.sort_values(
                grouping_columns + [ordering_column]
            ).reset_index(drop=True)

            for feature_name in numeric_feature_columns:
                working_dataframe[feature_name] = (
                    working_dataframe
                    .groupby(grouping_columns, dropna=False)[feature_name]
                    .ffill()
                )
    
        for feature_name in numeric_feature_columns:
            median_value = float(stats_dataframe[feature_name].median(skipna=True))
            working_dataframe[feature_name] = working_dataframe[feature_name].fillna(median_value)

        return working_dataframe, {
            "imputation_method": method,
            "grouping_columns": grouping_columns,
            "ordering_column": ordering_column,
        }

    # Fill Method - Global Mean
    if method == "global_mean":
        for feature_name in numeric_feature_columns:
            mean_value = float(stats_dataframe[feature_name].mean(skipna=True))
            working_dataframe[feature_name] = working_dataframe[feature_name].fillna(mean_value)

        return working_dataframe, {
            "imputation_method": method,
            "grouping_columns": [],
            "ordering_column": None,
        }

    for feature_name in numeric_feature_columns:
        median_value = float(stats_dataframe[feature_name].median(skipna=True))
        working_dataframe[feature_name] = working_dataframe[feature_name].fillna(median_value)

    return working_dataframe, {
        "imputation_method": "global_median",
        "grouping_columns": [],
        "ordering_column": None,
    }

In [ ]:
# Build a mask Series for fitting imputation stats
train_mask_for_stats = gold_working_dataframe["meta__is_train_flag"].astype(bool)

# Impute using train-only stats, applied to all rows
gold_working_dataframe, imputation_info = apply_imputation(
    gold_working_dataframe,
    numeric_feature_columns=numeric_feature_columns,
    method="forward_fill_within_group_then_median",
    train_mask=train_mask_for_stats,
)



#### #### #### #### #### #### #### #### 

# TODO: Need Logger
# TODO: Need Ledger

#### #### #### #### #### #### #### #### 

In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# Rebuild a fresh Series mask from the stamped column 
# We have to do this because the index may have changed after imputation

train_mask_flag = gold_working_dataframe["meta__is_train_flag"].astype(bool)


#### #### #### #### #### #### #### #### 

# TODO: Need Logger
# TODO: Need Ledger

#### #### #### #### #### #### #### #### 



In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
gold_preprocessed_prescaled_dataframe = gold_working_dataframe.copy()

gold_preprocessed_prescaled_path = save_data(
    gold_preprocessed_prescaled_dataframe,
    GOLD_PRESCALED_DATA_PATH.parent,
    GOLD_PRESCALED_DATA_PATH.name,
)

#### #### #### #### #### #### #### #### 

wandb.save(str(gold_preprocessed_prescaled_path))

#### #### #### #### #### #### #### #### 

ledger.add(
    kind="step",
    step="save_gold_preprocessed_prescaled",
    message="Saved full Gold preprocessed prior to scaling dataset.",
    data={
        "gold_preprocessed_prescaled_path": str(gold_preprocessed_prescaled_path),
        "gold_preprocessed_prescale_shape": list(gold_preprocessed_prescaled_dataframe.shape),
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 


In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
def make_scaler(kind: str = "robust"):
    kind = kind.lower()
    if kind == "standard":
        return StandardScaler()
    elif kind == "minmax":
        return MinMaxScaler()
    elif kind == "robust":
        return RobustScaler()
    else:
        raise ValueError(f"Unknown scaler kind: {kind}. Use 'standard', 'minmax', or 'robust'.")

In [ ]:
def fit_and_apply_scaler(
    dataframe: pd.DataFrame,
    feature_columns: Sequence[str],
    train_mask: pd.Series,
    normal_only_mask: Optional[pd.Series],
    scaler_kind: str,
    artifacts_path: Path,
    dataset_name: str,
    ledger,
) -> Tuple[pd.DataFrame, Path]:
    """
    Fit a scaler on normal-only train rows and apply it to all rows.

    Parameters
    ----------
    dataframe : pd.DataFrame
        Gold working dataframe BEFORE scaling.
    feature_columns : list[str] or Sequence[str]
        Numeric feature columns to scale.
    train_mask : pd.Series[bool]
        Boolean mask indicating train rows (True = train).
    normal_only_mask : pd.Series[bool] or None
        Boolean mask indicating "normal behavior" rows.
        If None, scaler is fit on all train rows.
    scaler_kind : str
        "robust", "standard", or "minmax".
    artifacts_path : Path
        Directory where artifacts (scaler joblib, etc.) should be saved.
    dataset_name : str
        Dataset identifier used in artifact filenames.
    ledger : Ledger
        Your custom ledger instance for logging the step.

    Returns
    -------
    scaled_dataframe : pd.DataFrame
        Copy of dataframe with scaled features in-place.
    scaler_path : Path
        Location of the saved scaler joblib file.
    """


    # Determine which rows to use for fitting (train ∩ normal-only)
    if normal_only_mask is not None:
        fit_mask = train_mask & normal_only_mask
        fit_source = "train ∩ normal-only"
    else:
        fit_mask = train_mask
        fit_source = "train (no explicit normal-only mask)"

    fit_rows = dataframe.loc[fit_mask, feature_columns]

    if fit_rows.empty:
        raise ValueError(
            "No rows available to fit the scaler. "
            "Check your train_mask and normal_only_mask."
        )


    # Make and fit the scaler
    scaler = make_scaler(scaler_kind=scaler_kind)
    scaler.fit(fit_rows.values)

    # Apply the scaler to ALL rows for the given feature_columns
    scaled_dataframe = dataframe.copy()
    scaled_dataframe.loc[:, feature_columns] = scaler.transform(
        scaled_dataframe[feature_columns].values
    )

    # Update Meta Stamp
    scaled_dataframe["meta__gold_scaler"] = str(scaler_kind)

    # Save scaler artifact

    artifacts_path.mkdir(parents=True, exist_ok=True)
    
    scaler_filename = SCALER_ARTIFACT_NAME_TEMPLATE.format(
        dataset=dataset_name,
        scaler_kind=scaler_kind.lower(),
    )

    scaler_path = artifacts_path / scaler_filename
    joblib.dump(scaler, scaler_path)



    #### ### #### #### #### ### #### #### 

    # Log to ledger
    ledger.add(
        kind="transform",
        step="gold_scaling",
        params={
            "scaler_kind": scaler_kind.lower(),
            "fit_source": fit_source,
            "fit_row_count": int(fit_rows.shape[0]),
            "feature_count": int(len(feature_columns)),
            "feature_columns": list(feature_columns),
        },
        artifacts={
            "scaler_path": str(scaler_path),
        },
    )

    # Log to Weights and Biases
    if wandb.run is not None:
        wandb.config.update(
            {
                "gold_scaler.kind": scaler_kind.lower(),
                "gold_scaler.feature_count": int(len(feature_columns)),
            },
            allow_val_change=True,
        )
        wandb.log(
            {
                "gold_scaler.fit_rows": int(fit_rows.shape[0]),
            }
        )

        #### ### #### #### #### ### #### #### 

    return scaled_dataframe, scaler_path

In [ ]:

gold_working_dataframe["meta__gold_imputation"] = imputation_info["imputation_method"]
gold_working_dataframe["meta__gold_scaler"] = "none"

#### #### #### #### #### #### #### #### 

normal_only_mask = (
    (gold_working_dataframe["anomaly_flag"] == 0)
    if "anomaly_flag" in gold_working_dataframe.columns
    else None
)

gold_preprocessed_scaled_dataframe, scaler_path = fit_and_apply_scaler(
    dataframe=gold_working_dataframe,
    feature_columns=numeric_feature_columns,
    train_mask=train_mask_flag,
    normal_only_mask=normal_only_mask,
    scaler_kind=SCALER_KIND,
    artifacts_path=GOLD_ARTIFACTS_PATH,
    dataset_name=DATASET_NAME,
    ledger=ledger,
)



print(f"Scaler saved to: {scaler_path}")
print(f"Scaled dataframe shape: {gold_preprocessed_scaled_dataframe.shape}")

#### #### #### #### #### #### #### #### 

gold_build_info = {
    "numeric_feature_count": int(len(numeric_feature_columns)),
    "feature_set_id": str(feature_set_id),
    "imputation": gold_preprocessed_scaled_dataframe["meta__gold_imputation"].iloc[0],
    "scaler": gold_preprocessed_scaled_dataframe["meta__gold_scaler"].iloc[0],
}

#### #### #### #### #### #### #### #### 

ledger.add(
    kind="step",
    step="build_gold_model_ready_dataframe",
    message="Built Gold model-ready dataframe using Silver feature registry and Silver EDA imputation recommendation.",
    data=gold_build_info,
    logger=logger,
)

#### #### #### #### #### #### #### #### 

gold_build_info

2026-03-03 01:02:24,365 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-03T01:02:24.365591+00:00', 'stage': 'gold', 'recipe': 'gold_preprocessing__v001_cascade', 'kind': 'step', 'step': 'build_gold_model_ready_dataframe', 'message': 'Built Gold model-ready dataframe using Silver feature registry and Silver EDA imputation recommendation.', 'why': None, 'consequence': None, 'data': {'numeric_feature_count': 50, 'feature_set_id': 'feature_set__bea5fdd737ae53fcd80ca84cafcd0c40', 'imputation': 'forward_fill_within_group_then_median', 'scaler': 'none'}}


{'numeric_feature_count': 50,
 'feature_set_id': 'feature_set__bea5fdd737ae53fcd80ca84cafcd0c40',
 'imputation': 'forward_fill_within_group_then_median',
 'scaler': 'none'}

In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
gold_preprocessed_scaled_path = save_data(
    gold_preprocessed_scaled_dataframe,
    GOLD_SCALED_DATA_PATH.parent,
    GOLD_SCALED_DATA_PATH.name,
)

#### #### #### #### #### #### #### #### 

wandb.save(str(gold_preprocessed_scaled_path))

#### #### #### #### #### #### #### #### 

ledger.add(
    kind="step",
    step="save_gold_preprocessed",
    message="Saved full Gold preprocessed dataset.",
    data={
        "gold_preprocessed_scaled_path": str(gold_preprocessed_prescaled_path),
        "gold_preprocessed_scaled_shape": list(gold_preprocessed_scaled_dataframe.shape),
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 


2026-03-03 01:02:25,023 | INFO | capstone.file_io | Saving DataFrame to Parquet: /workspace/data/gold/pump__gold__preprocessed.parquet
2026-03-03 01:02:27,935 | INFO | capstone.file_io | Saved: pump__gold__preprocessed.parquet | shape=(220320, 87) | columns=['meta__asset_id', 'meta__cleaning_recipe_id', 'meta__dataset', 'meta__dataset_name', 'meta__dataset_source', 'meta__event_id', 'meta__event_time_parse_success_percent', 'meta__event_time_source', 'meta__feature_count', 'meta__feature_set_id', 'meta__has_label_candidates', 'meta__has_status_candidates', 'meta__ingested_at_utc', 'meta__label_source', 'meta__label_source_column', 'meta__label_source_kind', 'meta__label_type', 'meta__layer', 'meta__processed_at_utc', 'meta__record_id', 'meta__run_id', 'meta__silver_version', 'meta__source_file', 'meta__source_row_id', 'meta__split', 'event_time', 'event_step', 'time_index', 'event_date', 'anomaly_flag', 'is_anomaly', 'is_normal', 'status_normal_value', 'sensor_00', 'sensor_01', 'sensor

{'ts_utc': '2026-03-03T01:02:27.965322+00:00',
 'stage': 'gold',
 'recipe': 'gold_preprocessing__v001_cascade',
 'kind': 'step',
 'step': 'save_gold_preprocessed',
 'message': 'Saved full Gold preprocessed dataset.',
 'why': None,
 'consequence': None,
 'data': {'gold_preprocessed_path': '/workspace/data/gold/pump__gold__preprocessed.parquet',
  'gold_shape': [220320, 87]}}

In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
def get_training_rows_for_unsupervised_model(
    dataframe: pd.DataFrame,
    *,
    train_mask: pd.Series,
) -> pd.DataFrame:
    training_subset = dataframe.loc[train_mask].copy()

    if "anomaly_flag" in training_subset.columns:
        training_subset = training_subset[training_subset["anomaly_flag"] == 0].copy()

    return training_subset

In [ ]:
# Normal Only
training_rows_for_fit = get_training_rows_for_unsupervised_model(
    gold_preprocessed_scaled_dataframe,
    train_mask=train_mask_flag,
)


#### #### #### #### #### #### #### #### 

# TODO: Need Logger
# TODO: Need Ledger

#### #### #### #### #### #### #### #### 


In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
def build_reference_profile(
    dataframe: pd.DataFrame,
    *,
    feature_columns: list[str],
) -> pd.DataFrame:
    reference_rows: list[dict] = []

    for feature_name in feature_columns:
        feature_series = dataframe[feature_name]

        reference_rows.append({
            "feature_name": feature_name,
            "median_value": float(feature_series.median()),
            "mean_value": float(feature_series.mean()),
            "standard_deviation": float(feature_series.std()) if not pd.isna(feature_series.std()) else 0.0,
            "lower_bound": float(feature_series.quantile(0.05)),
            "upper_bound": float(feature_series.quantile(0.95)),
        })

    reference_profile = pd.DataFrame(reference_rows)
    
    return reference_profile

In [ ]:
reference_profile = build_reference_profile(
    training_rows_for_fit,
    feature_columns=numeric_feature_columns,
)


#### #### #### #### #### #### #### #### 

# TODO: Need Logger
# TODO: Need Ledger

#### #### #### #### #### #### #### #### 


In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
def choose_stage2_features_from_training_stability(
    training_dataframe: pd.DataFrame,
    *,
    feature_columns: list[str],
    target_count: int,
) -> list[str]:
    ranking_rows: list[dict] = []

    for feature_name in feature_columns:
        feature_series = training_dataframe[feature_name]
        median_value = float(feature_series.median())
        standard_deviation = float(feature_series.std()) if not pd.isna(feature_series.std()) else 0.0

        coefficient_of_variation = standard_deviation / max(abs(median_value), 1e-6)

        ranking_rows.append({
            "feature_name": feature_name,
            "median_value": median_value,
            "standard_deviation": standard_deviation,
            "coefficient_of_variation": coefficient_of_variation,
        })

    ranking_dataframe = pd.DataFrame(ranking_rows)

    ranking_dataframe = ranking_dataframe.sort_values(
        by=["coefficient_of_variation", "standard_deviation", "feature_name"],
        ascending=[True, True, True]
    ).reset_index(drop=True)

    chosen_features = ranking_dataframe["feature_name"].head(target_count).tolist()
    
    return chosen_features

In [ ]:
stage1_feature_columns = list(numeric_feature_columns)

stage2_feature_columns = choose_stage2_features_from_training_stability(
    training_rows_for_fit,
    feature_columns=stage1_feature_columns,
    target_count=STAGE2_TARGET_FEATURE_COUNT,
)


#### #### #### #### #### #### #### #### 

# TODO: Need Logger
# TODO: Need Ledger

#### #### #### #### #### #### #### #### 


In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
def build_stage3_sensor_groups(
    reference_profile: pd.DataFrame,
    *,
    stage2_feature_columns: list[str],
    primary_count: int,
    secondary_count: int,
) -> tuple[list[str], list[str]]:
    ranked_reference = reference_profile[reference_profile["feature_name"].isin(stage2_feature_columns)].copy()

    ranked_reference = ranked_reference.sort_values(
        by=["standard_deviation", "feature_name"],
        ascending=[True, True]
    ).reset_index(drop=True)

    primary_rule_sensors = ranked_reference["feature_name"].head(primary_count).tolist()

    remaining_features = [
        feature_name
        for feature_name in ranked_reference["feature_name"].tolist()
        if feature_name not in primary_rule_sensors
    ]

    secondary_rule_sensors = remaining_features[:secondary_count]

    return primary_rule_sensors, secondary_rule_sensors

In [ ]:
stage3_primary_rule_sensors, stage3_secondary_rule_sensors = build_stage3_sensor_groups(
    reference_profile,
    stage2_feature_columns=stage2_feature_columns,
    primary_count=STAGE3_PRIMARY_COUNT,
    secondary_count=STAGE3_SECONDARY_COUNT,
)


#### #### #### #### #### #### #### #### 

# TODO: Need Logger
# TODO: Need Ledger

#### #### #### #### #### #### #### #### 


In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
save_json(stage1_feature_columns, STAGE1_FEATURES_PATH)
save_json(stage2_feature_columns, STAGE2_FEATURES_PATH)
save_json(stage3_primary_rule_sensors, STAGE3_PRIMARY_PATH)
save_json(stage3_secondary_rule_sensors, STAGE3_SECONDARY_PATH)

#### #### #### ####  

wandb.save(str(STAGE1_FEATURES_PATH))
wandb.save(str(STAGE2_FEATURES_PATH))
wandb.save(str(STAGE3_PRIMARY_PATH))
wandb.save(str(STAGE3_SECONDARY_PATH))

#### #### #### ####  

feature_set_summary = {
    "stage1_feature_count": int(len(stage1_feature_columns)),
    "stage2_feature_count": int(len(stage2_feature_columns)),
    "stage3_primary_rule_count": int(len(stage3_primary_rule_sensors)),
    "stage3_secondary_rule_count": int(len(stage3_secondary_rule_sensors)),
}

#### #### #### #### #### #### #### #### 

ledger.add(
    kind="step",
    step="build_stage_feature_sets",
    message="Built feature sets for baseline, Stage 1, Stage 2, and Stage 3.",
    data=feature_set_summary,
    logger=logger,
)

#### #### #### #### #### #### #### #### 

feature_set_summary    

2026-03-03 01:02:36,182 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__stage1_features.json
2026-03-03 01:02:36,208 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__stage2_features.json
2026-03-03 01:02:36,227 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__stage3_primary_rule_sensors.json
2026-03-03 01:02:36,248 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__stage3_secondary_rule_sensors.json
2026-03-03 01:02:36,416 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-03T01:02:36.416136+00:00', 'stage': 'gold', 'recipe': 'gold_preprocessing__v001_cascade', 'kind': 'step', 'step': 'build_stage_feature_sets', 'message': 'Built feature sets for baseline, Stage 1, Stage 2, and Stage 3.', 'why': None, 'consequence': None, 'data': {'stage1_feature_count': 50, 'stage2_feature_count': 12, 'stage3_primary_rule_count': 5, 'stage3_secondary_rule_count': 7}

{'stage1_feature_count': 50,
 'stage2_feature_count': 12,
 'stage3_primary_rule_count': 5,
 'stage3_secondary_rule_count': 7}

In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

## Save Gold Split Artifacts

This step saves the Gold preprocessing outputs as separate parquet artifacts so downstream modeling notebooks can use a fixed and reproducible split.

The notebook already created a time-ordered split earlier in the workflow:

- The **train split** contains the earlier portion of the time-ordered Gold dataset.
- The **test split** contains the later holdout portion of the time-ordered Gold dataset.
- The **fit subset** contains only normal rows from the training split and is used for unsupervised model fitting.

Saving these artifacts ensures that the baseline, cascade, and comparison notebooks all use the same consistent partitions.

In [ ]:
gold_train_dataframe = gold_preprocessed_scaled_dataframe.loc[train_mask_flag].copy()
gold_test_dataframe  = gold_preprocessed_scaled_dataframe.loc[~train_mask_flag].copy()
gold_fit_dataframe   = training_rows_for_fit.copy()

#### #### #### ####  

gold_train_path = save_data(
    gold_train_dataframe,
    GOLD_TRAIN_DATA_PATH.parent,
    GOLD_TRAIN_DATA_PATH.name,
)

gold_test_path = save_data(
    gold_test_dataframe,
    GOLD_TEST_DATA_PATH.parent,
    GOLD_TEST_DATA_PATH.name,
)

gold_fit_path = save_data(
    gold_fit_dataframe,
    GOLD_FIT_DATA_PATH.parent,
    GOLD_FIT_DATA_PATH.name,
)

#### #### #### #### 

wandb.save(str(gold_train_path))
wandb.save(str(gold_test_path))
wandb.save(str(gold_fit_path))

#### #### #### ####  

gold_split_summary = {
    "train_rows": int(len(gold_train_dataframe)),
    "test_rows": int(len(gold_test_dataframe)),
    "fit_rows_normal_only": int(len(gold_fit_dataframe)),
    "train_path": str(gold_train_path),
    "test_path": str(gold_test_path),
    "fit_path": str(gold_fit_path),
}

#### #### #### #### #### #### #### #### 

ledger.add(
    kind="step",
    step="save_gold_split_artifacts",
    message="Saved Gold train, test, and normal-only fit parquet artifacts.",
    data=gold_split_summary,
    logger=logger,
)

#### #### #### #### #### #### #### #### 

pd.DataFrame(
    [
        {
            "split": "train",
            "rows": int(len(gold_train_dataframe)),
            "abnormal_rows": int(gold_train_dataframe["anomaly_flag"].sum()) if "anomaly_flag" in gold_train_dataframe.columns else None,
            "path": str(gold_train_path),
        },
        {
            "split": "test",
            "rows": int(len(gold_test_dataframe)),
            "abnormal_rows": int(gold_test_dataframe["anomaly_flag"].sum()) if "anomaly_flag" in gold_test_dataframe.columns else None,
            "path": str(gold_test_path),
        },
        {
            "split": "fit_normal_only",
            "rows": int(len(gold_fit_dataframe)),
            "abnormal_rows": int(gold_fit_dataframe["anomaly_flag"].sum()) if "anomaly_flag" in gold_fit_dataframe.columns else None,
            "path": str(gold_fit_path),
        },
    ]
)
#### #### #### #### #### #### #### #### 


2026-03-03 01:02:38,904 | INFO | capstone.file_io | Saving DataFrame to Parquet: /workspace/data/gold/pump__gold__train.parquet
2026-03-03 01:02:41,077 | INFO | capstone.file_io | Saved: pump__gold__train.parquet | shape=(154224, 87) | columns=['meta__asset_id', 'meta__cleaning_recipe_id', 'meta__dataset', 'meta__dataset_name', 'meta__dataset_source', 'meta__event_id', 'meta__event_time_parse_success_percent', 'meta__event_time_source', 'meta__feature_count', 'meta__feature_set_id', 'meta__has_label_candidates', 'meta__has_status_candidates', 'meta__ingested_at_utc', 'meta__label_source', 'meta__label_source_column', 'meta__label_source_kind', 'meta__label_type', 'meta__layer', 'meta__processed_at_utc', 'meta__record_id', 'meta__run_id', 'meta__silver_version', 'meta__source_file', 'meta__source_row_id', 'meta__split', 'event_time', 'event_step', 'time_index', 'event_date', 'anomaly_flag', 'is_anomaly', 'is_normal', 'status_normal_value', 'sensor_00', 'sensor_01', 'sensor_02', 'sensor_

,split,rows,abnormal_rows,path
0,train,154224,14408,/workspace/data/gold/pump__gold__train.parquet
1,test,66096,76,/workspace/data/gold/pump__gold__test.parquet
2,fit_normal_only,139816,0,/workspace/data/gold/pump__gold__fit_normal_on...


In [ ]:
#scaler = locals().get("scaler", None)

#### #### #### #### #### #### #### #### 

preprocessing_summary = {
    "gold_path": str(GOLD_DATA_PATH),
    "gold_scaled_shape": list(gold_preprocessed_scaled_dataframe.shape),
    "feature_count": int(len(numeric_feature_columns)),
    "stage1_feature_count": int(len(stage1_feature_columns)),
    "stage2_feature_count": int(len(stage2_feature_columns)),
    "stage3_primary_rule_count": int(len(stage3_primary_rule_sensors)),
    "stage3_secondary_rule_count": int(len(stage3_secondary_rule_sensors)),
    "imputation_method": str(imputation_info.get("imputation_method")),
    "grouping_columns": list(imputation_info.get("grouping_columns", [])),
    "ordering_column": imputation_info.get("ordering_column"),
    "scaler_kind": str(SCALER_KIND),
    "train_fraction": float(TRAIN_FRACTION),
    "train_rows": int(len(gold_train_dataframe)),
    "test_rows": int(len(gold_test_dataframe)),
    "fit_rows_normal_only": int(len(gold_fit_dataframe)),
    "gold_train_path": str(gold_train_path),
    "gold_test_path": str(gold_test_path),
    "gold_fit_path": str(gold_fit_path),
}

#### #### #### #### #### #### #### #### 

save_json(preprocessing_summary, PREPROCESSING_SUMMARY_PATH)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

preprocessing_metadata = {
    "recipe_id": RECIPE_ID,
    "gold_version": GOLD_VERSION,
    "dataset_name": DATASET_NAME,
    "feature_set_source": str(FEATURE_REGISTRY_PATH),
    "imputation_recommendation_source": str(IMPUTE_RECOMMENDATION_PATH),
    "gold_output_path": str(GOLD_DATA_PATH),
    "reference_profile_path": str(REFERENCE_PROFILE_PATH),
    "stage1_features_path": str(STAGE1_FEATURES_PATH),
    "stage2_features_path": str(STAGE2_FEATURES_PATH),
    "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
    "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
    "preprocessor_scaler_path": str(scaler_path) if SCALER_KIND else None,
    "gold_train_path": str(gold_train_path),
    "gold_test_path": str(gold_test_path),
    "gold_fit_path": str(gold_fit_path),
}

#### #### #### #### #### #### #### #### 

save_json(preprocessing_metadata, PREPROCESSING_METADATA_PATH)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

reference_profile.to_csv(REFERENCE_PROFILE_PATH, index=False)

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

'''
# Already saved in function 

if SCALER_KIND and scaler is not None:
    joblib.dump(scaler, scaler_path)


wandb.save(str(PREPROCESSING_SUMMARY_PATH))
wandb.save(str(PREPROCESSING_METADATA_PATH))
wandb.save(str(REFERENCE_PROFILE_PATH))

if SCALER_KIND and scaler is not None:
    wandb.save(str(scaler_path))

'''

#### #### #### #### #### #### #### #### 

ledger.add(
    kind="step",
    step="save_preprocessing_outputs",
    message="Saved Gold preprocessing outputs, metadata, and reference profile artifacts.",
    data={
        "preprocessing_summary_path": str(PREPROCESSING_SUMMARY_PATH),
        "preprocessing_metadata_path": str(PREPROCESSING_METADATA_PATH),
        "reference_profile_path": str(REFERENCE_PROFILE_PATH),
        "preprocessor_scaler_path": str(scaler_path) if SCALER_KIND else None,
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 

ledger.add(
    kind="step",
    step="finalize_preprocessing",
    message="Gold preprocessing notebook complete.",
    data={
        "gold_path": str(GOLD_DATA_PATH),
        "stage1_features_path": str(STAGE1_FEATURES_PATH),
        "stage2_features_path": str(STAGE2_FEATURES_PATH),
        "stage3_primary_path": str(STAGE3_PRIMARY_PATH),
        "stage3_secondary_path": str(STAGE3_SECONDARY_PATH),
        "gold_scaled_shape": list(gold_preprocessed_scaled_dataframe.shape),
        "feature_count": int(len(numeric_feature_columns)),
        "stage1_feature_count": int(len(stage1_feature_columns)),
        "stage2_feature_count": int(len(stage2_feature_columns)),
        "stage3_primary_rule_count": int(len(stage3_primary_rule_sensors)),
        "stage3_secondary_rule_count": int(len(stage3_secondary_rule_sensors)),
    },
    logger=logger,
)

#### #### #### #### #### #### #### #### 

preprocessing_ledger_path = GOLD_ARTIFACTS_PATH / GOLD_PREPROCESSING_LEDGER_FILE_NAME
ledger.write_json(preprocessing_ledger_path)

#### #### #### #### #### #### #### #### 

wandb.save(str(preprocessing_ledger_path))
wandb_run.finish()

2026-03-03 01:02:44,039 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__preprocessing_summary.json
2026-03-03 01:02:44,077 | INFO | capstone.file_io | Saved JSON: /workspace/artifacts/gold/pump/pump__gold__preprocessing_metadata.json
2026-03-03 01:02:44,460 | INFO | capstone.gold | LEDGER | {'ts_utc': '2026-03-03T01:02:44.460010+00:00', 'stage': 'gold', 'recipe': 'gold_preprocessing__v001_cascade', 'kind': 'step', 'step': 'save_preprocessing_outputs', 'message': 'Saved Gold preprocessing outputs, metadata, and reference profile artifacts.', 'why': None, 'consequence': None, 'data': {'preprocessing_summary_path': '/workspace/artifacts/gold/pump/pump__gold__preprocessing_summary.json', 'preprocessing_metadata_path': '/workspace/artifacts/gold/pump/pump__gold__preprocessing_metadata.json', 'reference_profile_path': '/workspace/artifacts/gold/pump/pump__gold__reference_profile.csv', 'preprocessor_scaler_path': None}}
2026-03-03 01:02:44,464 | INFO | capst

In [ ]:
#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 

In [ ]:
# -------------------------------------------------------------------
# Gold Preprocessing Sanity Checks
# -------------------------------------------------------------------

print("Running Gold preprocessing sanity checks...\n")

# 1) Basic existence & shape checks
assert "gold_preprocessed_scaled_dataframe" in locals(), "gold_preprocessed_scaled_dataframe is missing."
assert "train_mask_flag" in locals(), "train_mask_flag is missing."
assert "numeric_feature_columns" in locals(), "numeric_feature_columns is missing."
assert "gold_train_dataframe" in locals(), "gold_train_dataframe is missing."
assert "gold_test_dataframe" in locals(), "gold_test_dataframe is missing."
assert "gold_fit_dataframe" in locals(), "gold_fit_dataframe is missing."

df_scaled = gold_preprocessed_scaled_dataframe
n_rows, n_cols = df_scaled.shape

print(f"- Scaled Gold shape: {n_rows} rows x {n_cols} cols")

# 2) Train/test masks line up with splits
train_mask_flag = train_mask_flag.astype(bool)

assert len(train_mask_flag) == n_rows, "train_mask_flag length does not match Gold dataframe."
assert len(gold_train_dataframe) == train_mask_flag.sum(), "Train split row count mismatch."
assert len(gold_test_dataframe) == (~train_mask_flag).sum(), "Test split row count mismatch."

print(f"- Train rows: {len(gold_train_dataframe)}, Test rows: {len(gold_test_dataframe)}")

# 3) Fit (normal-only) subset sanity check
if "anomaly_flag" in df_scaled.columns:
    # Fit rows should be train & anomaly_flag == 0
    fit_mask_expected = train_mask_flag & (df_scaled["anomaly_flag"] == 0)
    assert len(gold_fit_dataframe) == fit_mask_expected.sum(), "Fit (normal-only) row count mismatch."
    assert gold_fit_dataframe["anomaly_flag"].sum() == 0, "Fit dataframe contains anomalous rows."
    print(f"- Fit (normal-only) rows: {len(gold_fit_dataframe)} (anomaly_flag == 0 confirmed)")
else:
    print("- anomaly_flag not present, skipping normal-only checks for fit dataframe.")

# 4) Meta columns sanity
for col in ["meta__is_train_flag", "meta__gold_imputation", "meta__gold_scaler"]:
    assert col in df_scaled.columns, f"Expected meta column '{col}' is missing."

print(f"- Meta columns present: meta__is_train_flag, meta__gold_imputation, meta__gold_scaler")
print(f"- Imputation method: {df_scaled['meta__gold_imputation'].iloc[0]}, scaler: {df_scaled['meta__gold_scaler'].iloc[0]}")

# 5) Check that numeric features actually changed by scaling
if "gold_preprocessed_prescaled_dataframe" in locals():
    df_prescaled = gold_preprocessed_prescaled_dataframe
    assert df_prescaled.shape == df_scaled.shape, "Prescaled & scaled shapes differ."

    diffs = []
    for col in numeric_feature_columns:
        if col not in df_prescaled.columns:
            continue
        # Compare on a small sample to keep it cheap
        sample_scaled = df_scaled[col].head(100).to_numpy()
        sample_prescaled = df_prescaled[col].head(100).to_numpy()
        if not np.allclose(sample_scaled, sample_prescaled):
            diffs.append(col)

    assert len(diffs) > 0, "No numeric features appear to have changed after scaling."
    print(f"- Scaling changed {len(diffs)} numeric feature(s) (good).")
else:
    print("- gold_preprocessed_prescaled_dataframe not found; skipping pre vs post scaling comparison.")

# 6) Check scaler behavior on train ∩ normal-only
if "scaler_path" in locals() and Path(scaler_path).exists():
    loaded_scaler = joblib.load(scaler_path)

    # Build the fit mask we *expect* was used
    if "anomaly_flag" in df_scaled.columns:
        normal_only_mask = (df_scaled["anomaly_flag"] == 0)
        fit_mask_expected = train_mask_flag & normal_only_mask
        fit_source_desc = "train ∩ normal-only"
    else:
        fit_mask_expected = train_mask_flag
        fit_source_desc = "train (no anomaly_flag)"

    fit_rows_expected = df_scaled.loc[fit_mask_expected, numeric_feature_columns]
    assert len(fit_rows_expected) > 0, "Expected fit rows for scaler are empty."

    # Check that transforming those rows matches what is already in the dataframe
    # (i.e., df_scaled == scaler.transform(original) for those rows)
    # This is lossy because we don't have the *pre*-scaled values here, but we can at least sanity check shape.
    transformed_sample = loaded_scaler.transform(fit_rows_expected.values)
    assert transformed_sample.shape == fit_rows_expected[numeric_feature_columns].values.shape, \
        "Scaler transform output shape mismatch for fit rows."

    print(f"- Scaler artifact loaded from: {scaler_path}")
    print(f"- Scaler was configured to fit on: {fit_source_desc}")
else:
    print("- scaler_path missing or file not found; skipping scaler artifact check.")

# 7) Basic distribution checks for scaled training data, per scaler kind
SCALER_KIND_LOWER = str(SCALER_KIND).lower()

train_norm = gold_fit_dataframe[numeric_feature_columns]
summary_checks = []

if SCALER_KIND_LOWER == "standard":
    means = train_norm.mean()
    stds = train_norm.std(ddof=1)
    mean_close = (means.abs() < 0.5).mean()  # fraction of features with |mean| < 0.5
    std_close = ((stds - 1).abs() < 0.5).mean()  # fraction with std ~ 1
    summary_checks.append(f"StandardScaler: ~{mean_close:.2%} of features have |mean| < 0.5; "
                          f"~{std_close:.2%} have std within 0.5 of 1.")

elif SCALER_KIND_LOWER == "robust":
    medians = train_norm.median()
    q75 = train_norm.quantile(0.75)
    q25 = train_norm.quantile(0.25)
    iqr = q75 - q25
    median_close = (medians.abs() < 0.5).mean()
    iqr_close = ((iqr - 1).abs() < 0.75).mean()  # allow a bit more tolerance
    summary_checks.append(f"RobustScaler: ~{median_close:.2%} of features have |median| < 0.5; "
                          f"~{iqr_close:.2%} have IQR within 0.75 of 1.")

elif SCALER_KIND_LOWER == "minmax":
    mins = train_norm.min()
    maxs = train_norm.max()
    min_ok = (mins >= -1e-3).mean()
    max_ok = (maxs <= 1 + 1e-3).mean()
    summary_checks.append(f"MinMaxScaler: ~{min_ok:.2%} of features have min ≥ 0; "
                          f"~{max_ok:.2%} have max ≤ 1.")

for msg in summary_checks:
    print("- " + msg)

print("\nAll Gold preprocessing sanity checks completed.")